# Continuous Audit — Planner Notifier
Cria um card no bucket STAND-BY para cada novo achado/reincidente. Se o último
card do teste ainda está aberto, atualiza em vez de duplicar (trilha em
`tb_notification_log`). Credenciais Graph nos secrets `planner-*` do scope
`compliance-grc`. Consumido pelo orquestrador via `%run` após o `utils`.


In [ ]:
import uuid
import requests
from datetime import datetime
from zoneinfo import ZoneInfo
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

_BRT   = ZoneInfo("America/Sao_Paulo")
_GRAPH = "https://graph.microsoft.com/v1.0"

# ── Config do Planner (resolvida por NOME a cada rodada — sobrevive a mudanças de ID)
_PLANNER_GROUP_ID  = "b5d2a8c9-e93c-4f88-b09b-0aca8f9c7147"   # grupo GRC
_PLANNER_PLAN_NAME = "Planner_GRC"
_PLANNER_BUCKET    = "STAND-BY"                                # cards SEMPRE nascem aqui
_LABEL_HINTS       = ("gestão de riscos", "dedo no pulso")     # labels rosa + roxa

_CHECKLIST = [
    "Analisar os achados no painel e marcar falsos positivos, se houver",
    "Acionar a área responsável e definir o plano de ação",
    "Se for a tratamento: emitir e vincular o apontamento no painel — o alerta fica suprimido até o apontamento fechar",
    "Reavaliar após as ações e confirmar rodada sem achados",
]

_ALERT_LABEL = {"novo_achado": "Novo achado", "reincidente": "Reincidente"}


def _graph_headers():
    tenant = dbutils.secrets.get("compliance-grc", "planner-tenant-id")
    cid    = dbutils.secrets.get("compliance-grc", "planner-client-id")
    csec   = dbutils.secrets.get("compliance-grc", "planner-client-secret")
    r = requests.post(
        f"https://login.microsoftonline.com/{tenant}/oauth2/v2.0/token",
        data={"client_id": cid, "client_secret": csec,
              "grant_type": "client_credentials",
              "scope": "https://graph.microsoft.com/.default"},
        timeout=20,
    ).json()
    if "access_token" not in r:
        raise RuntimeError(f"Token Graph falhou: {str(r)[:200]}")
    return {"Authorization": f"Bearer {r['access_token']}",
            "Content-Type": "application/json"}


def _resolve_plan(H):
    plans   = requests.get(f"{_GRAPH}/groups/{_PLANNER_GROUP_ID}/planner/plans",
                           headers=H, timeout=20).json()["value"]
    plan    = next(p for p in plans if p["title"] == _PLANNER_PLAN_NAME)
    buckets = requests.get(f"{_GRAPH}/planner/plans/{plan['id']}/buckets",
                           headers=H, timeout=20).json()["value"]
    bucket  = next(b for b in buckets
                   if b["name"].strip().upper() == _PLANNER_BUCKET.upper())
    cats    = (requests.get(f"{_GRAPH}/planner/plans/{plan['id']}/details",
                            headers=H, timeout=20).json()
               .get("categoryDescriptions") or {})
    labels  = {k: True for k, v in cats.items()
               if v and any(h in v.lower() for h in _LABEL_HINTS)}
    return plan["id"], bucket["id"], (labels or {"category1": True, "category6": True})


# ── Log de notificações: trilha + base do dedup por episódio ─────────────────
def _ensure_log_table():
    """Best-effort: o SP do job tem CREATE TABLE e cria na 1ª execução; um
    usuário sem a permissão não deve explodir aqui (a tabela pode já existir)."""
    try:
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.tb_notification_log (
                log_id STRING, test_name STRING, alert_type STRING,
                planner_task_id STRING, incident_count INT,
                created_at TIMESTAMP, updated_at TIMESTAMP
            ) USING DELTA
        """)
    except Exception as e:
        print(f"Planner: não consegui garantir tb_notification_log ({str(e)[:120]})")


def _latest_task_id(test_name):
    safe = test_name.replace("'", "''")
    try:
        rows = spark.sql(f"""
            SELECT planner_task_id FROM {CATALOG}.{SCHEMA}.tb_notification_log
            WHERE test_name = '{safe}' ORDER BY created_at DESC LIMIT 1
        """).collect()
        return rows[0]["planner_task_id"] if rows else None
    except Exception:
        return None  # tabela ainda não existe → sem histórico de cards


def _task_is_open(H, task_id):
    r = requests.get(f"{_GRAPH}/planner/tasks/{task_id}", headers=H, timeout=20)
    if r.status_code != 200:
        return False  # excluída → episódio novo
    return r.json().get("percentComplete", 100) < 100


def _description(e, risk, app_url):
    linhas = ["Trigger da Auditoria Contínua — achados que exigem análise.", "",
              f"Teste: {e['test_name']}"]
    if e.get("description"):
        linhas.append(f"O que o teste verifica: {e['description']}")
    if e.get("risco_id") and e["risco_id"] != "N/A":
        extra = ""
        if risk.get("title"):
            extra += f" — {risk['title']}"
        if risk.get("level"):
            extra += f" (Inerente: {risk['level']})"
        linhas.append(f"Risco: {e['risco_id']}{extra}")
    if e.get("area"):
        linhas.append(f"Área responsável: {e['area']}")
    linhas.append(f"Tipo de alerta: {_ALERT_LABEL.get(e['alert'], e['alert'])}")
    linhas.append(f"Achados na rodada: {e['count']}")
    linhas.append(f"Rodada: {datetime.now(_BRT).strftime('%d/%m/%Y %H:%M')} (BRT)")
    if app_url:
        linhas += ["", f"Painel: {app_url}"]
    return "\n".join(linhas)


def _log_schema():
    return StructType([
        StructField("log_id", StringType()), StructField("test_name", StringType()),
        StructField("alert_type", StringType()), StructField("planner_task_id", StringType()),
        StructField("incident_count", IntegerType()),
        StructField("created_at", TimestampType()), StructField("updated_at", TimestampType()),
    ])


def notify_planner_cards(events, risk_info=None, app_url=None) -> int:
    """Cria/atualiza cards no Planner para novos achados e reincidentes.

    Dedup exigido pelo processo: se o último card do teste ainda está ABERTO
    (percentComplete < 100), o card é atualizado — nunca duplicado. Card
    concluído pelo time (ou excluído) → novo trigger abre card novo.
    Erros de execução não viram card (problema técnico, não risco).
    """
    risk_info = risk_info or {}
    dedup = {}
    for e in events:
        dedup[e["test_name"]] = e
    cards = [e for e in dedup.values()
             if e["alert"] in ("novo_achado", "reincidente") and e["notify"]]
    _ensure_log_table()   # também em rodadas sem trigger — o job cria a tabela cedo
    if not cards:
        print("Planner: nenhum trigger para card.")
        return 0

    H = _graph_headers()
    plan_id, bucket_id, labels = _resolve_plan(H)

    criados = atualizados = 0
    for e in cards:
        risk = risk_info.get(e.get("risco_id"), {})
        desc = _description(e, risk, app_url)

        task_id = _latest_task_id(e["test_name"])
        if task_id and _task_is_open(H, task_id):
            det = requests.get(f"{_GRAPH}/planner/tasks/{task_id}/details",
                               headers=H, timeout=20)
            requests.patch(f"{_GRAPH}/planner/tasks/{task_id}/details",
                           headers={**H, "If-Match": det.json()["@odata.etag"]},
                           json={"description": desc}, timeout=20)
            safe = e["test_name"].replace("'", "''")
            try:
                spark.sql(f"""
                    UPDATE {CATALOG}.{SCHEMA}.tb_notification_log
                    SET incident_count = {int(e['count'])}, updated_at = current_timestamp()
                    WHERE planner_task_id = '{task_id}'
                """)
            except Exception as le:
                print(f"Planner: log NÃO atualizado para {e['test_name']} "
                      f"(permissão? {str(le)[:100]})")
            atualizados += 1
            print(f"Planner ↻ card aberto atualizado: {e['test_name']}")
            continue

        t = requests.post(f"{_GRAPH}/planner/tasks", headers=H, json={
            "planId": plan_id, "bucketId": bucket_id,
            "title": f"[Continuous Audit] {e['test_name']}",
            "priority": 3,
            "appliedCategories": labels,
        }, timeout=20).json()
        if "id" not in t:
            print(f"Planner: falha ao criar card de {e['test_name']}: {str(t)[:150]}")
            continue
        det = requests.get(f"{_GRAPH}/planner/tasks/{t['id']}/details",
                           headers=H, timeout=20)
        checklist = {str(i): {"@odata.type": "microsoft.graph.plannerChecklistItem",
                              "title": s, "isChecked": False}
                     for i, s in enumerate(_CHECKLIST, 1)}
        requests.patch(f"{_GRAPH}/planner/tasks/{t['id']}/details",
                       headers={**H, "If-Match": det.json()["@odata.etag"]},
                       json={"description": desc, "checklist": checklist,
                             "previewType": "checklist"}, timeout=20)
        agora = datetime.now(_BRT)
        try:
            (spark.createDataFrame(
                [(str(uuid.uuid4()), e["test_name"], e["alert"], t["id"],
                  int(e["count"]), agora, agora)], schema=_log_schema())
             .write.mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.tb_notification_log"))
        except Exception as le:
            print(f"Planner: ATENÇÃO — card de {e['test_name']} criado mas NÃO "
                  f"registrado no log ({str(le)[:100]}). O dedup ficará cego para "
                  f"este card; registre manualmente ou rode com permissão de escrita.")
        criados += 1
        print(f"Planner + card criado: {e['test_name']}")

    print(f"Planner: {criados} criado(s) · {atualizados} atualizado(s).")
    return criados
